# Find closest sequence

Typically, ClinVar uses only one or a few reference sequences for a human gene.  We
should find the sequences from other species that are most similar to the human
reference sequence.  

To illustrate, let's consider ABCD1.  Below we see that there are 2,067 variants
in the ClinVar database. 

**Note:** I had to update the code for the `get_variant_details()` function.
The `@RecordType` appears to have changed to `classified`.  I suspect that this
is due to the expansion of the database to include both germline and somatic
mutations.  The new function needs more comprehensive testing but it seems to
work fine on the ABCD1 variants.

In [176]:
run kondrashov

### Step 1 identify reference human sequence
##### I check each Clinvar variant per gene in 'pathogenic' directory and sift througth to identify unique transcripts

In [177]:
import pandas as pd
from tqdm import tqdm

ABCD1_df = pd.read_csv("pathogenic/ALPL.csv")
varids = ABCD1_df["VariationID"].astype(str).tolist()

unique_acc = set()

for varid in tqdm(varids):
    varrec = get_variant(varid)
    name, acc, mut, pmut, muttype, status = get_variant_details(varrec)

    if acc is not None:
        unique_acc.add(acc)

print(f"Found {len(unique_acc)} unique accession(s):")
print(sorted(unique_acc))

100%|██████████| 404/404 [06:12<00:00,  1.08it/s]

Found 1 unique accession(s):
['NM_000478.6']


### STEP 2 Obtain corresponding protein of each unique human transcript

In [178]:
transcript_record = get_transcript("NM_000478.6")
xx= get_protein_from_transcript(transcript_record)
xx

(Seq('atgatttcaccattcttagtactggccattggcacctgccttactaactcctta...tga'),
 Seq('MISPFLVLAIGTCLTNSLVPEKEKDPKYWRDQAQETLKYALELQKLNTNVAKNV...VLF'),
 'NP_000469.3',
 199,
 1774)

### This code combines step 1 and step 2
#### ignore 

In [393]:
import pandas as pd
from tqdm import tqdm
all_proteins = {}

loci = ["ALPL", "ABCD1", "CFTR"]
# Read ClinVar variants
for gene in loci:
    df = pd.read_csv(f"pathogenic/{gene}.csv")
    varids = df["VariationID"].astype(str).tolist()
    unique_acc = set()
    
    for varid in tqdm(varids):
        varrec = get_variant(varid)
        name, acc, mut, pmut, muttype, status = get_variant_details(varrec)
        
        if acc is not None:
            unique_acc.add(acc)
    print(f"Found {len(unique_acc)} unique transcript accession(s):")
    print(sorted(unique_acc))

# Retrieve transcripts and extract proteins
protein_records = []

#for transcript in tqdm(unique_acc):
for transcript in tqdm(sorted(unique_acc), desc=f"{gene}: transcripts"):
    transcript_record = get_transcript(transcript)
    protein = get_protein_from_transcript(transcript_record)
            
    protein_records.append({
        "gene": gene,
        "transcript": transcript,
        "protein": protein
    })
    all_proteins[gene] = protein_records
# View results
for record in protein_records:
    print(record["gene"], record["transcript"], record["protein"])

100%|██████████| 404/404 [05:27<00:00,  1.23it/s]


Found 1 unique transcript accession(s):
['NM_000478.6']


100%|██████████| 327/327 [04:10<00:00,  1.31it/s]


Found 1 unique transcript accession(s):
['NM_000033.4']


 55%|█████▌    | 428/773 [05:58<04:48,  1.20it/s]


IndexError: list index out of range

## Run these next two codes instead
### These combine step 1 and step 2 using functions 'identify_unique_transcripts' and 'get_reference_proteins'

In [395]:
run kondrashov

In [396]:
#loci = ["ALPL", "ABCD1", "CFTR"]    
gene_transcripts = {}

for gene in loci:
    gene_transcripts[gene] = identify_unique_transcripts(gene)

ABCD1: variants:   0%|          | 0/327 [00:00<?, ?it/s]

ABCD1: variants: 100%|██████████| 327/327 [05:19<00:00,  1.02it/s]


ABCD1: Found 1 unique transcript(s)


ALPL: variants: 100%|██████████| 404/404 [05:40<00:00,  1.19it/s]


ALPL: Found 1 unique transcript(s)


...

TP53: Found 1 unique transcript(s)


TTR: variants: 100%|██████████| 130/130 [01:05<00:00,  1.99it/s]


TTR: Found 1 unique transcript(s)


VWF: variants: 100%|██████████| 310/310 [02:30<00:00,  2.06it/s]

VWF: Found 1 unique transcript(s)


In [400]:
gene_transcripts.items()

dict_items([('ABCD1', {'NM_000033.4'}), ('ALPL', {'NM_000478.6'}), ('AR', {'NM_000044.6'}), ('ATP7B', {'NM_000053.4'}), ('BTK', {'NM_000061.3'}), ('CASR', {'NM_000388.4'}), ('CBS', {'NM_000071.3'}), ('CFTR', {'NM_000492.3', 'NM_000492.4'}), ('CYBB', {'NM_000397.4'}), ('F7', {'NM_000131.5', 'NM_000131.4', 'NM_019616.4'}), ('F8', {'NM_000132.4'}), ('F9', {'NM_000133.4'}), ('G6PD', {'NM_001360016.2', 'NM_001042351.3', 'NM_000402.4'}), ('GALT', {'NM_000155.4'}), ('GBA1', {'NM_000157.4'}), ('GJB1', {'NM_001097642.3', 'NM_000166.6'}), ('HBB', {'NM_000518.5', 'NM_000518.4'}), ('HPRT1', {'NM_000194.2', 'NM_000194.3'}), ('IL2RG', {'NM_000206.3'}), ('KCNH2', {'NM_000238.4'}), ('KCNQ1', {'NM_000218.3'}), ('L1CAM', {'NM_001278116.2'}), ('LDLR', {'NM_000527.4', 'NM_000527.5'}), ('MPZ', {'NM_000530.8'}), ('MYH7', {'NM_000257.4'}), ('TYR', {'NM_000372.4', 'NM_000372.5'}), ('PAH', {'NM_000277.3'}), ('PMM2', {'NM_000303.3'}), ('RHO', {'NM_000539.3'}), ('TP53', {'NM_000546.6'}), ('TTR', {'NM_000371.4'})

In [401]:
reference_proteins = {}

for gene, transcripts in gene_transcripts.items():
    reference_proteins[gene] = []
    for transcript in sorted(transcripts):
        prot = get_reference_proteins(gene, transcript)
        reference_proteins[gene].append(prot)
        print(gene, prot)

ABCD1 [{'transcript': 'NM_000033.4', 'protein_id': 'NP_000024.2', 'protein_seq': Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'), 'cds_seq': Seq('atgccggtgctctccaggccccggccctggcgggggaacacgctgaagcgcacg...tga'), 'cds_start': 411, 'cds_end': 2649}]
ALPL [{'transcript': 'NM_000478.6', 'protein_id': 'NP_000469.3', 'protein_seq': Seq('MISPFLVLAIGTCLTNSLVPEKEKDPKYWRDQAQETLKYALELQKLNTNVAKNV...VLF'), 'cds_seq': Seq('atgatttcaccattcttagtactggccattggcacctgccttactaactcctta...tga'), 'cds_start': 199, 'cds_end': 1774}]
AR [{'transcript': 'NM_000044.6', 'protein_id': 'NP_000035.2', 'protein_seq': Seq('MEVQLGLGRVYPRPPSKTYRGAFQNLFQSVREVIQNPGPRHPEAASAAPPGASL...HTQ'), 'cds_seq': Seq('atggaagtgcagttagggctgggaagggtctaccctcggccgccgtccaagacc...tga'), 'cds_start': 1126, 'cds_end': 3889}]
ATP7B [{'transcript': 'NM_000053.4', 'protein_id': 'NP_000044.2', 'protein_seq': Seq('MPEQERQITAREGASRKILSKLSLPTRAWEPAMKKSFAFDNVGYEGGLDGLGPS...QYI'), 'cds_seq': Seq('atgcctgagcaggagagacagatcacagccagagaaggg

In [402]:
reference_proteins

{'ABCD1': [[{'transcript': 'NM_000033.4',
    'protein_id': 'NP_000024.2',
    'protein_seq': Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'),
    'cds_seq': Seq('atgccggtgctctccaggccccggccctggcgggggaacacgctgaagcgcacg...tga'),
    'cds_start': 411,
    'cds_end': 2649}]],
 'ALPL': [[{'transcript': 'NM_000478.6',
    'protein_id': 'NP_000469.3',
    'protein_seq': Seq('MISPFLVLAIGTCLTNSLVPEKEKDPKYWRDQAQETLKYALELQKLNTNVAKNV...VLF'),
    'cds_seq': Seq('atgatttcaccattcttagtactggccattggcacctgccttactaactcctta...tga'),
    'cds_start': 199,
    'cds_end': 1774}]],
 'AR': [[{'transcript': 'NM_000044.6',
    'protein_id': 'NP_000035.2',
    'protein_seq': Seq('MEVQLGLGRVYPRPPSKTYRGAFQNLFQSVREVIQNPGPRHPEAASAAPPGASL...HTQ'),
    'cds_seq': Seq('atggaagtgcagttagggctgggaagggtctaccctcggccgccgtccaagacc...tga'),
    'cds_start': 1126,
    'cds_end': 3889}]],
 'ATP7B': [[{'transcript': 'NM_000053.4',
    'protein_id': 'NP_000044.2',
    'protein_seq': Seq('MPEQERQITAREGASRKILSKLSLP

In [431]:
reference_proteins['HPRT1'][1]

[{'transcript': 'NM_000194.3',
  'protein_id': 'NP_000185.1',
  'protein_seq': Seq('MATRSPGVVISDDEPGYDLDLFCIPNHYAEDLERVFIPHGLIMDRTERLARDVM...YKA'),
  'cds_seq': Seq('atggcgacccgcagccctggcgtcgtgattagtgatgatgaaccaggttatgac...taa'),
  'cds_start': 147,
  'cds_end': 804}]

In [ ]:

## ignore
loci = ["ALPL", "ABCD1", "CFTR"]
for gene in loci:
    for transcript in gene_transcripts[gene]:
        #print(transcript)
        prot = get_reference_proteins(gene, transcript)
        print(gene, prot)


ALPL [{'transcript': 'NM_000478.6', 'protein_id': 'NP_000469.3', 'protein_seq': Seq('MISPFLVLAIGTCLTNSLVPEKEKDPKYWRDQAQETLKYALELQKLNTNVAKNV...VLF'), 'cds_seq': Seq('atgatttcaccattcttagtactggccattggcacctgccttactaactcctta...tga'), 'cds_start': 199, 'cds_end': 1774}]
ABCD1 [{'transcript': 'NM_000033.4', 'protein_id': 'NP_000024.2', 'protein_seq': Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'), 'cds_seq': Seq('atgccggtgctctccaggccccggccctggcgggggaacacgctgaagcgcacg...tga'), 'cds_start': 411, 'cds_end': 2649}]
CFTR [{'transcript': 'NM_000492.3', 'protein_id': 'NP_000483.3', 'protein_seq': Seq('MQRSPLEKASVVSKLFFSWTRPILRKGYRQRLELSDIYQIPSVDSADNLSEKLE...TRL'), 'cds_seq': Seq('atgcagaggtcgcctctggaaaaggccagcgttgtctccaaactttttttcagc...tag'), 'cds_start': 132, 'cds_end': 4575}]
CFTR [{'transcript': 'NM_000492.4', 'protein_id': 'NP_000483.3', 'protein_seq': Seq('MQRSPLEKASVVSKLFFSWTRPILRKGYRQRLELSDIYQIPSVDSADNLSEKLE...TRL'), 'cds_seq': Seq('atgcagaggtcgcctctggaaaaggccagcgttgtctcc

## Ignore
### Step 3 Read through each fasta file in 'fasta/' folder
#### Identify unique sequences

In [408]:
run kondrashov

In [422]:
ortholog_data = {}

for gene in loci:

    all_sequences = []
    seq_records = []
    species_names = []

    for seq_record in SeqIO.parse(f"fasta/{gene}.fasta", "fasta"):

        seq = seq_record.seq
        species = "[" + seq_record.description.split("[")[-1]

        if seq not in all_sequences:
            all_sequences.append(seq)
            seq_records.append(seq_record.id)
            species_names.append(species)

    ortholog_data[gene] = {
        "all_sequences": all_sequences,
        "seq_records": seq_records,
        "species_names": species_names,
    }
    #print(gene, seq_record.id, len(seq), species, sep="\t")

In [424]:
print(ortholog_data)
ortholog_data

{'ABCD1': {'all_sequences': [Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'), Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'), Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...VTR'), Seq('MDRLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYVRYRLPLRVVKHFY...LSQ'), Seq('MDMTPGLRHTGQSMDRLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYV...LSQ'), Seq('MDMTPGLRHTGQSMARLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYV...LSQ'), Seq('MARLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYVRYRLPLRVVKHFY...LSQ'), Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPARGSQAPAREPT...AST'), Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPARGSQAPAREPT...WPM'), Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPARGSQAPAREPT...AGL'), Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'), Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...AST'), Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...AGL'), Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...WPM'), Seq('

{'ABCD1': {'all_sequences': [Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'),
   Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'),
   Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...VTR'),
   Seq('MDRLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYVRYRLPLRVVKHFY...LSQ'),
   Seq('MDMTPGLRHTGQSMDRLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYV...LSQ'),
   Seq('MDMTPGLRHTGQSMARLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYV...LSQ'),
   Seq('MARLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYVRYRLPLRVVKHFY...LSQ'),
   Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPARGSQAPAREPT...AST'),
   Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPARGSQAPAREPT...WPM'),
   Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPARGSQAPAREPT...AGL'),
   Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'),
   Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...AST'),
   Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...AGL'),
   Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVH

In [ ]:
## Ignore

protein_Record = {}

for gene in loci: 
    print(f'{gene} orthologs: all species sequences\n')
    all_sequences = []
    seq_records = []
    species_names = []

    inc = 0
    exc = 0
    for seq_record in SeqIO.parse(f'fasta/{gene}.fasta', "fasta"):
        #sp = get_species(seq_record)
        seq = seq_record.seq
        species = "[" + seq_record.description.split("[")[-1]
        if seq in all_sequences:
            print("\t{0}\t({1} aa)\t{2}\t*** the same as sequence {3} (excluded) ***".format(seq_record.id, len(seq), seq_record.description, all_sequences.index(seq)))
            exc += 1
        else:
            all_sequences.append(seq)
            species_names.append(species)
            seq_records.append(seq_record)        
            print(inc, seq_record.id, len(seq), species, sep= '\t')
            inc += 1


In [ ]:
all_sequences

### STEP 4   Finding the most similar *species* sequence to the reference human sequence

The following defines a pairwise aligner object using the Needleman-Wunsch
algorithm using the default `blastp` scoring scheme and substitution matrix.

In [205]:
from Bio import SeqIO, Align
from Bio.Seq import Seq

In [206]:
aligner = Align.PairwiseAligner(scoring="blastp")
print(aligner)

Pairwise sequence aligner with parameters
  substitution_matrix: <Array object at 0x1a7e721a0b0>
  open_internal_insertion_score: -12.000000
  extend_internal_insertion_score: -1.000000
  open_left_insertion_score: -12.000000
  extend_left_insertion_score: -1.000000
  open_right_insertion_score: -12.000000
  extend_right_insertion_score: -1.000000
  open_internal_deletion_score: -12.000000
  extend_internal_deletion_score: -1.000000
  open_left_deletion_score: -12.000000
  extend_left_deletion_score: -1.000000
  open_right_deletion_score: -12.000000
  extend_right_deletion_score: -1.000000
  mode: global



In [207]:
print(aligner.substitution_matrix[:, :])

     A    B    C    D    E    F    G    H    I    J    K    L    M    N    O    P    Q    R    S    T    U    V    W    X    Y    Z    *
A  4.0 -2.0  0.0 -2.0 -1.0 -2.0  0.0 -2.0 -1.0 -1.0 -1.0 -1.0 -1.0 -2.0 -1.0 -1.0 -1.0 -1.0  1.0  0.0  0.0  0.0 -3.0 -1.0 -2.0 -1.0 -4.0
B -2.0  4.0 -3.0  4.0  1.0 -3.0 -1.0  0.0 -3.0 -3.0  0.0 -4.0 -3.0  4.0 -1.0 -2.0  0.0 -1.0  0.0 -1.0 -3.0 -3.0 -4.0 -1.0 -3.0  0.0 -4.0
C  0.0 -3.0  9.0 -3.0 -4.0 -2.0 -3.0 -3.0 -1.0 -1.0 -3.0 -1.0 -1.0 -3.0 -1.0 -3.0 -3.0 -3.0 -1.0 -1.0  9.0 -1.0 -2.0 -1.0 -2.0 -3.0 -4.0
D -2.0  4.0 -3.0  6.0  2.0 -3.0 -1.0 -1.0 -3.0 -3.0 -1.0 -4.0 -3.0  1.0 -1.0 -1.0  0.0 -2.0  0.0 -1.0 -3.0 -3.0 -4.0 -1.0 -3.0  1.0 -4.0
E -1.0  1.0 -4.0  2.0  5.0 -3.0 -2.0  0.0 -3.0 -3.0  1.0 -3.0 -2.0  0.0 -1.0 -1.0  2.0  0.0  0.0 -1.0 -4.0 -2.0 -3.0 -1.0 -2.0  4.0 -4.0
F -2.0 -3.0 -2.0 -3.0 -3.0  6.0 -3.0 -1.0  0.0  0.0 -3.0  0.0  0.0 -3.0 -1.0 -4.0 -3.0 -3.0 -2.0 -2.0 -2.0 -1.0  1.0 -1.0  3.0 -3.0 -4.0
G  0.0 -1.0 -3.0 -1.0 -2.0 -3.0  6.0 -2.0

In [ ]:
max_score = aligner.score(hum_sequences[0], hum_sequences[0])
for i in range(50):
    score = aligner.score(hum_sequences[0], all_sequences[i])
    print(f"{seq_ids}: {score/max_score:.4f}")

In [330]:
max_score = aligner.score(hum_sequences[0], hum_sequences[0])

for species, seq_record, seq in zip(species_names, seq_records,  all_sequences):
    score = aligner.score(hum_sequences[0], seq)
    print(f"{seq_record.id}: {species}: {score/max_score:.4f}")

NP_000469.3: [Homo sapiens]: 1.0000
NP_001120973.2: [Homo sapiens]: 0.8740
NP_001170991.1: [Homo sapiens]: 0.7839
XP_016856392.1: [Homo sapiens]: 0.8418
XP_054191723.1: [Homo sapiens]: 0.8400
NP_001253798.1: [Macaca mulatta]: 0.9786
XP_014985684.2: [Macaca mulatta]: 0.9783
XP_014985703.1: [Macaca mulatta]: 0.8219
XP_077852312.1: [Macaca mulatta]: 0.8526
XP_077852318.1: [Macaca mulatta]: 0.7639
XP_016811306.1: [Pan troglodytes]: 0.9906
XP_016811335.1: [Pan troglodytes]: 0.8874
XP_016811339.1: [Pan troglodytes]: 0.8660
NP_001171963.1: [Callithrix jacchus]: 0.9667
XP_008998771.1: [Callithrix jacchus]: 0.8675
XP_005544582.2: [Macaca fascicularis]: 0.9772
XP_065384427.1: [Macaca fascicularis]: 0.9461
XP_069335566.1: [Eulemur rufifrons]: 0.9316
XP_069335568.1: [Eulemur rufifrons]: 0.7256
XP_055118431.1: [Symphalangus syndactylus]: 0.9801
XP_054402715.1: [Pongo abelii]: 0.9866
XP_054402718.1: [Pongo abelii]: 0.8834
XP_054402719.1: [Pongo abelii]: 0.8606
XP_054303319.1: [Pongo pygmaeus]: 0.987

## This automates step 4
##### The function 'write_best_species_matches' extracts the highest-scoring sequence per unique species from the fasta file (fasta/{gene}.fasta) and writes a new file (fasta/{gene}_match.fasta)

In [337]:
run kondrashov

In [ ]:
best = write_best_species_matches(gene="ALPL", reference_seq=hum_sequences[0], aligner=aligner)

31 sequences written to fasta/ALPL_match.fasta


In [339]:
best

{'[Homo sapiens]': {'record': SeqRecord(seq=Seq('MISPFLVLAIGTCLTNSLVPEKEKDPKYWRDQAQETLKYALELQKLNTNVAKNV...VLF'), id='NP_000469.3', name='NP_000469.3', description='NP_000469.3 alkaline phosphatase, tissue-nonspecific isozyme isoform 1 preproprotein [Homo sapiens]', dbxrefs=[]),
  'score': 1.0},
 '[Macaca mulatta]': {'record': SeqRecord(seq=Seq('MISPFLVLAIGTCLTNSLVPEKEKDPKYWRDQAQETLKYALELQKLNTNVAKNV...ILF'), id='NP_001253798.1', name='NP_001253798.1', description='NP_001253798.1 alkaline phosphatase, tissue-nonspecific isozyme precursor [Macaca mulatta]', dbxrefs=[]),
  'score': 0.9786386676321506},
 '[Pan troglodytes]': {'record': SeqRecord(seq=Seq('MISPFLVLAIGTCLTNSLVPEKEKDPKYWRDQAQETLKYALKLQKLNTNVAKNV...VLF'), id='XP_016811306.1', name='XP_016811306.1', description='XP_016811306.1 alkaline phosphatase, tissue-nonspecific isozyme isoform X1 [Pan troglodytes]', dbxrefs=[]),
  'score': 0.9905865314989138},
 '[Callithrix jacchus]': {'record': SeqRecord(seq=Seq('MISPFLVLAIGTCLTNSLVPEKEKDP

In [ ]:
best = write_best_species_matches(gene= gene, reference_seq=hum_sequences[0], aligner=aligner)

In [437]:
for ref in reference_proteins['F9']:
    transcript = ref[0]["transcript"]
    protein_id = ref[0]["protein_id"]
    reference_seq = ref[0]["protein_seq"]
print(f"  {transcript} ({protein_id})")

  NM_000133.4 (NP_000124.1)


In [ ]:
loci = ['ALPL', 'F8', 'F9']
for gene in loci:

    print(f"\nProcessing {gene}")

    for ref in reference_proteins[gene]:

        transcript = ref[0]["transcript"]
        protein_id = ref[0]["protein_id"]
        reference_seq = ref[0]["protein_seq"]

        print(f"  {transcript} ({protein_id})")

        write_best_species_matches(
            gene=gene,
            reference_seq=ref["protein_seq"],
            aligner=aligner,
            outfile=f"fasta/{gene}_{ref['transcript']}_match.fasta"
        )


Processing ALPL
  NM_000478.6 (NP_000469.3)


TypeError: list indices must be integers or slices, not str